# Results

This chapter reports the performance of the evaluated forecasting strategies under the common chronological split and matched data design. The main comparison is followed by analyses of negative transfer, robustness, source filtering, and selector behaviour.


In [ ]:
import os
from pathlib import Path

candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(path for path in candidate_roots if (path / "Data" / "processed").exists())
MPLCONFIG_PATH = PROJECT_ROOT / ".cache" / "matplotlib"
MPLCONFIG_PATH.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_PATH))

try:
    import matplotlib
    matplotlib.use("Agg")

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
except Exception as exc:
    raise RuntimeError(
        "This notebook needs a clean Python 3 kernel with numpy, pandas, and matplotlib. "
        "If your current Anaconda kernel shows a numpy or pyarrow error, switch kernels and run again."
    ) from exc

from IPython.display import Image, display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed"
METHOD_TABLES_DIR = PROCESSED_DIR / "methodology" / "tables"
EDA_TABLES_DIR = PROCESSED_DIR / "eda" / "tables"
RESULTS_FIG_DIR = PROCESSED_DIR / "results" / "figures"
RESULTS_TABLES_DIR = PROCESSED_DIR / "results" / "tables"
RESULTS_FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_TABLES_DIR.mkdir(parents=True, exist_ok=True)

EDA_TABLE_GROUPS = {
    "overview": {
        "corpus_overview.csv",
        "feature_missingness.csv",
        "split_audit.csv",
    },
    "correlations": {
        "pooled_lag_correlations.csv",
        "site_best_lag_summary.csv",
        "site_lag_correlations.csv",
    },
    "similarity": {
        "source_target_best_matches.csv",
        "source_target_best_matches_train.csv",
        "source_target_similarity.csv",
        "source_target_similarity_train.csv",
    },
}

METHOD_TABLE_GROUPS = {
    "protocol": {
        "protocol_summary.csv",
        "feature_inventory.csv",
        "similarity_summary.csv",
        "transfer_configuration.csv",
    },
    "core_results": {
        "summary_long.csv",
        "summary_wide.csv",
        "gain_vs_no_tl.csv",
        "site_metric_long.csv",
        "site_metric_wide.csv",
        "best_model_by_site.csv",
        "publication_core_comparison.csv",
    },
    "selector": {
        "selector_search.csv",
        "selector_by_site.csv",
        "selector_allocation.csv",
        "selector_feature_importance.csv",
        "site_hard_selector_by_site.csv",
    },
    "transfer": {
        "site_transfer_effects.csv",
        "transfer_effect_summary.csv",
        "cohort_definitions.csv",
        "cohort_assignments.csv",
        "cohort_pooled_summary.csv",
        "cohort_site_summary.csv",
        "negative_transfer_site_summary.csv",
        "negative_transfer_summary.csv",
        "negative_transfer_similarity_bins.csv",
    },
    "error_analysis": {
        "test_error_detail.csv",
        "error_by_anomaly_regime.csv",
        "error_by_signed_anomaly_regime.csv",
        "error_by_calendar_month.csv",
        "site_error_diagnostics.csv",
        "selector_failure_typology.csv",
        "error_driver_correlations.csv",
        "worst_site_error_profile.csv",
    },
    "robustness": {
        "selector_ablation_summary.csv",
        "paired_significance_tests.csv",
        "cluster_bootstrap_significance.csv",
        "label_budget_detail.csv",
        "label_budget_summary.csv",
        "seed_stability_detail.csv",
        "seed_stability_summary.csv",
        "rolling_temporal_folds.csv",
        "rolling_temporal_detail.csv",
        "rolling_temporal_summary.csv",
        "publication_robustness_comparison.csv",
    },
}

EDA_TABLE_DIRS = {group: EDA_TABLES_DIR / group for group in EDA_TABLE_GROUPS}
METHOD_TABLE_DIRS = {group: METHOD_TABLES_DIR / group for group in METHOD_TABLE_GROUPS}

EDA_TABLE_PATHS = {
    filename: EDA_TABLE_DIRS[group] / filename
    for group, filenames in EDA_TABLE_GROUPS.items()
    for filename in filenames
}
METHOD_TABLE_PATHS = {
    filename: METHOD_TABLE_DIRS[group] / filename
    for group, filenames in METHOD_TABLE_GROUPS.items()
    for filename in filenames
}

MODEL_COLORS = {
    "Context-Aware Selective Learning": "#0B6E4F",
    "Deep Context LSTM (Always TL)": "#C97C00",
    "Deep Context LSTM (No TL)": "#6B7280",
    "Random Forest": "#2C7FB8",
    "Baseline LSTM (No TL)": "#9AA5B1",
    "Baseline LSTM (TL)": "#B56576",
    "Site-Hard Expert Selector": "#8D6A9F",
}

SITE_EFFECT_COLORS = {
    "Selective rescue": "#0B6E4F",
    "Persistent harm": "#C0392B",
    "Both improve": "#2C7FB8",
    "Always TL only": "#9C6644",
}

FEATURE_GROUP_COLORS = {
    "Recent local state": "#0B6E4F",
    "Similarity profile": "#2C7FB8",
    "Seasonality": "#C97C00",
    "Site history": "#8D6A9F",
    "External drivers": "#9C6644",
    "Other": "#6B7280",
}


def methodology_table_path(name: str) -> Path:
    if name in METHOD_TABLE_PATHS:
        return METHOD_TABLE_PATHS[name]
    for directory in METHOD_TABLE_DIRS.values():
        candidate = directory / name
        if candidate.exists():
            return candidate
    return METHOD_TABLES_DIR / name


def eda_table_path(name: str) -> Path:
    if name in EDA_TABLE_PATHS:
        return EDA_TABLE_PATHS[name]
    for directory in EDA_TABLE_DIRS.values():
        candidate = directory / name
        if candidate.exists():
            return candidate
    return EDA_TABLES_DIR / name


def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def apply_thesis_style() -> None:
    plt.rcParams.update(
        {
            "figure.dpi": 160,
            "savefig.dpi": 300,
            "font.family": "DejaVu Serif",
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.facecolor": "#FBFBF8",
            "figure.facecolor": "white",
            "axes.grid": True,
            "grid.color": "#D7DBDD",
            "grid.linewidth": 0.6,
            "grid.alpha": 0.6,
            "axes.labelcolor": "#1F2933",
            "xtick.color": "#1F2933",
            "ytick.color": "#1F2933",
            "axes.edgecolor": "#A7B0B8",
        }
    )


def save_and_display(fig: plt.Figure, filename: str, width: int = 920) -> Path:
    out_path = RESULTS_FIG_DIR / filename
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(out_path), width=width))
    print(f"Saved: {out_path}")
    return out_path


def _humanize_feature(name: str) -> str:
    pretty = {
        "context_last_wtda": "Recent local groundwater depth",
        "month_sin": "Seasonality sin(month)",
        "month_cos": "Seasonality cos(month)",
        "wtda_mean": "Target well mean depth",
        "wtda_std": "Target well depth variability",
        "pr_a_mean": "Mean precipitation anomaly",
        "sm_a_mean": "Mean soil-moisture anomaly",
        "tsmp_wtda_std": "TSMP depth variability",
        "pumping_log_std": "Pumping variability",
        "lon_norm": "Longitude",
        "lat_norm": "Latitude",
        "sim_min": "Minimum source similarity",
        "sim_q90": "90th percentile similarity",
        "sim_q95": "95th percentile similarity",
        "sim_top3_mean": "Mean top-3 similarity",
        "sim_top5_mean": "Mean top-5 similarity",
        "sim_top10_mean": "Mean top-10 similarity",
        "train_rows": "Available training history",
    }
    if name in pretty:
        return pretty[name]
    if name.startswith("shared_last_"):
        remainder = name.removeprefix("shared_last_").replace("_", " ")
        return f"Shared recent {remainder}"
    if name.startswith("context_last_"):
        remainder = name.removeprefix("context_last_").replace("_", " ")
        return f"Recent local {remainder}"
    return name.replace("_", " ")


def _feature_group(name: str) -> str:
    if name.startswith("sim_") or "similarity" in name:
        return "Similarity profile"
    if name.startswith("month_"):
        return "Seasonality"
    if name.startswith("shared_last_") or name.startswith("context_last_"):
        return "Recent local state"
    if name.endswith("_mean") or name.endswith("_std") or name in {"train_rows", "max_observations"}:
        return "Site history"
    if any(token in name for token in ("pr_", "sm_", "tsmp_", "pumping", "lon_norm", "lat_norm")):
        return "External drivers"
    return "Other"


def _binned_mean(df: pd.DataFrame, xcol: str, ycol: str, n_bins: int = 6) -> pd.DataFrame:
    clean = df[[xcol, ycol]].dropna().copy()
    if clean.empty:
        return clean
    clean["bin"] = pd.qcut(clean[xcol], q=min(n_bins, clean[xcol].nunique()), duplicates="drop")
    grouped = (
        clean.groupby("bin", observed=True)
        .agg(x_mean=(xcol, "mean"), y_mean=(ycol, "mean"), n=(ycol, "size"))
        .reset_index(drop=True)
    )
    return grouped

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_FIG_DIR:", RESULTS_FIG_DIR)

## 1. Main Comparison

The first comparison focuses on held-out test performance under the fixed evaluation protocol.


In [ ]:
core_table = read_csv(methodology_table_path("publication_core_comparison.csv")).copy()
show_cols = [col for col in ["model_family", "rmse_test", "mae_test", "r2_test"] if col in core_table.columns]
core_display = core_table[show_cols].sort_values("rmse_test").reset_index(drop=True)
num_cols = core_display.select_dtypes(include=[np.number]).columns
core_display[num_cols] = core_display[num_cols].round(3)
display(core_display)


In [ ]:
apply_thesis_style()
core = read_csv(methodology_table_path("publication_core_comparison.csv")).copy()
core = core.sort_values("rmse_test", ascending=False)
colors = [MODEL_COLORS.get(model, "#6B7280") for model in core["model_family"]]
best_rmse = float(core["rmse_test"].min())

fig, ax = plt.subplots(figsize=(9.0, 5.0))
ax.barh(core["model_family"], core["rmse_test"], color=colors, edgecolor="white", linewidth=1.0)
ax.axvline(best_rmse, linestyle="--", linewidth=1.0, color="#1F2933", alpha=0.7)
ax.set_xlabel("Test RMSE")
ax.set_ylabel("")
fig.tight_layout()
main_comparison_path = save_and_display(fig, "results_main_comparison.png", width=900)


In the learned-model comparison, the conservative site-hard selective transfer policy has the lowest test RMSE (`2.045`), ahead of fixed transfer (`2.052`) by `0.0067` RMSE. The soft Context-Aware Selective Learning gate follows (`2.090`), then the no-transfer deep model (`2.133`) and Random Forest (`2.212`). I therefore frame the main result as evidence that a simple selective transfer policy can modestly improve fixed transfer, while the soft gate remains useful for analysing heterogeneous transfer effects.


## 2. Negative Transfer And Recovery

This section examines which wells benefit from transfer, which wells are harmed by transfer, and to what extent selective learning mitigates those harmful cases.


In [ ]:
apply_thesis_style()
site_effects = read_csv(methodology_table_path("site_transfer_effects.csv")).copy()
strategy_cols = [
    ("Always transfer", "delta_rmse_always_tl_vs_no_tl"),
    ("Soft selective learning", "delta_rmse_selective_tl_vs_no_tl"),
    ("Site-hard selector", "delta_rmse_site_hard_tl_vs_no_tl"),
]
plot_rows = []
for label, col in strategy_cols:
    values = site_effects[col].dropna()
    plot_rows.append(
        {
            "label": label,
            "helped_wells": int((values < 0).sum()),
            "neutral_wells": int((values == 0).sum()),
            "harmed_wells": int((values > 0).sum()),
        }
    )
plot_df = pd.DataFrame(plot_rows)

helped = plot_df["helped_wells"].to_numpy(dtype=float)
neutral = plot_df["neutral_wells"].to_numpy(dtype=float)
harmed = plot_df["harmed_wells"].to_numpy(dtype=float)
y_pos = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(8.6, 5.0))
ax.barh(y_pos, helped, color="#0B6E4F", label="Improved vs No TL")
ax.barh(y_pos, neutral, left=helped, color="#B9C0C7", label="Same as No TL")
ax.barh(y_pos, harmed, left=helped + neutral, color="#C0392B", label="Harmed vs No TL")
ax.set_yticks(y_pos, plot_df["label"])
ax.set_xlabel("Number of target wells")
ax.invert_yaxis()
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
fig.subplots_adjust(bottom=0.22)
transfer_counts_path = save_and_display(fig, "results_transfer_help_harm_counts.png", width=900)


The site-level help-harm comparison is the clearest motivation for selective transfer. Fixed transfer improves `57/99` wells but harms `42/99`; soft selective learning improves `60/99` and harms `39/99`; the conservative site-hard selector improves `43/99`, harms only `28/99`, and leaves `28/99` unchanged relative to the no-transfer parent. I therefore do not interpret transfer as a uniformly positive intervention in the Amsterdam setting. The contribution is negative-transfer mitigation: selective transfer gives up some fixed-transfer gains on easy wells in order to reduce damage where transfer is unreliable.


In [ ]:
apply_thesis_style()
site_effects = read_csv(methodology_table_path("site_transfer_effects.csv")).copy()
harmed_sites = site_effects[site_effects["delta_rmse_always_tl_vs_no_tl"] > 0].copy()
always_damage = harmed_sites["delta_rmse_always_tl_vs_no_tl"]
soft_damage = harmed_sites["delta_rmse_selective_tl_vs_no_tl"]
hard_damage = harmed_sites["delta_rmse_site_hard_tl_vs_no_tl"]

rescue_df = pd.DataFrame(
    [
        {
            "policy": "Soft selective learning",
            "partial_rescue_rate": float((soft_damage < always_damage).mean()),
            "full_rescue_rate": float((soft_damage <= 0).mean()),
        },
        {
            "policy": "Site-hard selector",
            "partial_rescue_rate": float((hard_damage < always_damage).mean()),
            "full_rescue_rate": float((hard_damage <= 0).mean()),
        },
    ]
)

fig, ax = plt.subplots(figsize=(7.4, 4.8))
x = np.arange(len(rescue_df))
width = 0.34
ax.bar(x - width / 2, rescue_df["partial_rescue_rate"] * 100.0, width=width, color="#2C7FB8", label="Partial rescue")
ax.bar(x + width / 2, rescue_df["full_rescue_rate"] * 100.0, width=width, color="#0B6E4F", label="Full rescue")
ax.set_xticks(x, rescue_df["policy"])
ax.set_ylim(0, 100)
ax.set_ylabel("Share of always-TL harmed wells (%)")
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
fig.subplots_adjust(bottom=0.24)
fig.tight_layout()
rescue_path = save_and_display(fig, "results_negative_transfer_rescue_rates.png", width=760)


Among the `42` wells harmed by fixed transfer, soft selective learning reduces the fixed-transfer damage for `97.6%` and fully rescues `9.5%`. The site-hard selector is more conservative: it fully rescues `33.3%` of harmed wells by falling back to the no-transfer parent, and it reduces the mean harmed-site RMSE penalty from `0.073` to `0.040`, a `45.8%` reduction. This is the strongest interpretation of the method: it mitigates negative transfer rather than producing a large universal accuracy gain.


In [ ]:
apply_thesis_style()
site_effects = read_csv(methodology_table_path("site_transfer_effects.csv")).copy()

def classify(row: pd.Series) -> str:
    x = float(row["delta_rmse_always_tl_vs_no_tl"])
    y = float(row["delta_rmse_selective_tl_vs_no_tl"])
    if x > 0 and y < 0:
        return "Selective rescue"
    if x > 0 and y >= 0:
        return "Persistent harm"
    if x <= 0 and y < 0:
        return "Both improve"
    return "Always TL only"

site_effects["regime"] = site_effects.apply(classify, axis=1)
site_effects["marker_size"] = np.clip(
    site_effects["months_used"].fillna(site_effects["months_used"].median()), 10, 110
) * 2.2

fig, ax = plt.subplots(figsize=(8.0, 6.3))
for regime, sub in site_effects.groupby("regime", observed=True):
    ax.scatter(
        sub["delta_rmse_always_tl_vs_no_tl"],
        sub["delta_rmse_selective_tl_vs_no_tl"],
        s=sub["marker_size"],
        alpha=0.8,
        color=SITE_EFFECT_COLORS[regime],
        edgecolor="white",
        linewidth=0.7,
        label=regime,
    )

bounds = np.array(
    [
        site_effects["delta_rmse_always_tl_vs_no_tl"].min(),
        site_effects["delta_rmse_always_tl_vs_no_tl"].max(),
        site_effects["delta_rmse_selective_tl_vs_no_tl"].min(),
        site_effects["delta_rmse_selective_tl_vs_no_tl"].max(),
    ]
)
pad = 0.03
lower, upper = bounds.min() - pad, bounds.max() + pad
ax.plot([lower, upper], [lower, upper], linestyle="--", color="#7F8C8D", linewidth=1.0, alpha=0.7)
ax.axhline(0, color="#1F2933", linewidth=0.9, alpha=0.7)
ax.axvline(0, color="#1F2933", linewidth=0.9, alpha=0.7)
ax.set_xlim(lower, upper)
ax.set_ylim(lower, upper)
ax.set_xlabel("Always TL - No TL (delta RMSE)")
ax.set_ylabel("Selective learning - No TL (delta RMSE)")
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
fig.subplots_adjust(bottom=0.24)
site_heterogeneity_path = save_and_display(fig, "results_site_transfer_heterogeneity.png", width=860)


The site-level comparison confirms strong heterogeneity. `56` wells improve under both transfer strategies, `4` are clear selective rescues, `38` remain harmed under both strategies, and `1` benefits only from fixed transfer. This is why the transfer claim should be framed as a negative-transfer diagnostic rather than as a blanket performance win.


## 3. Robustness

Robustness analyses are reported as thesis results rather than as supplementary checks. They are used to assess whether the advantage of selective learning persists under alternative resampling and evaluation settings.


In [ ]:
robustness_table = read_csv(methodology_table_path("publication_robustness_comparison.csv")).copy()
robustness_display = robustness_table.copy()
num_cols = robustness_display.select_dtypes(include=[np.number]).columns
robustness_display[num_cols] = robustness_display[num_cols].round(3)
display(robustness_display)

paired = read_csv(methodology_table_path("paired_significance_tests.csv")).copy()
paired = paired.query("model_a == 'Site-Hard Expert Selector' and analysis_unit == 'site_rmse'").reset_index(drop=True).copy()
paired_display = paired[[
    "model_b",
    "mean_delta_metric_a_minus_b",
    "wins_model_a",
    "wins_model_b",
    "p_value_holm",
]].copy()
paired_display.columns = [
    "comparison_model",
    "mean_site_rmse_delta",
    "wins_selective",
    "wins_comparison",
    "holm_p_value",
]
paired_display["mean_site_rmse_delta"] = paired_display["mean_site_rmse_delta"].round(4)
paired_display["holm_p_value"] = paired_display["holm_p_value"].round(6)
display(paired_display)


In [ ]:
apply_thesis_style()
bootstrap = read_csv(methodology_table_path("cluster_bootstrap_significance.csv")).copy()
label_map = {
    "Context-Aware Selective Learning": "Soft Selective",
    "Deep Context LSTM (Always TL)": "Always TL",
    "Deep Context LSTM (No TL)": "No TL",
    "Random Forest": "Random Forest",
}
bootstrap_df = bootstrap.query("model_a == 'Site-Hard Expert Selector'").copy()
bootstrap_df["comparison"] = bootstrap_df["model_b"].map(label_map).fillna(bootstrap_df["model_b"])
bootstrap_df = bootstrap_df.sort_values("mean_delta_site_rmse_a_minus_b")

y = np.arange(len(bootstrap_df))
mean_delta = bootstrap_df["mean_delta_site_rmse_a_minus_b"].to_numpy(dtype=float)
low = bootstrap_df["bootstrap_ci_lower"].to_numpy(dtype=float)
high = bootstrap_df["bootstrap_ci_upper"].to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(8.6, 4.8))
ax.hlines(y, low, high, color="#2C7FB8", linewidth=2.2)
ax.scatter(mean_delta, y, color="#0B6E4F", s=48, zorder=3)
ax.axvline(0, color="#1F2933", linewidth=0.9, alpha=0.7)
ax.set_yticks(y, bootstrap_df["comparison"])
ax.set_xlabel("Mean site RMSE delta")
fig.tight_layout()
bootstrap_path = save_and_display(fig, "results_cluster_bootstrap.png", width=900)


The cluster bootstrap gives a more cautious picture. The site-hard selector is clearly better than the baseline LSTMs and Random Forest, but its interval against fixed transfer overlaps zero. This means the main claim should be framed as selective transfer matching or slightly improving fixed transfer on the single held-out test split, not as a large statistically decisive gain over fixed transfer.


In [ ]:
apply_thesis_style()
seed_summary = read_csv(methodology_table_path("seed_stability_summary.csv")).copy()
focus_models = [
    "Site-Hard Expert Selector",
    "Context-Aware Selective Learning",
    "Deep Context LSTM (Always TL)",
    "Deep Context LSTM (No TL)",
    "Random Forest",
]
label_map = {
    "Site-Hard Expert Selector": "Site-Hard Selective",
    "Context-Aware Selective Learning": "Soft Selective",
    "Deep Context LSTM (Always TL)": "Always TL",
    "Deep Context LSTM (No TL)": "No TL",
    "Random Forest": "Random Forest",
}
seed_df = seed_summary.query("split == 'test' and model_family in @focus_models").copy()
seed_df["label"] = seed_df["model_family"].map(label_map)
seed_df["label"] = pd.Categorical(seed_df["label"], categories=["Site-Hard Selective", "Soft Selective", "Always TL", "No TL", "Random Forest"], ordered=True)
seed_df = seed_df.sort_values("label")

y_pos = np.arange(len(seed_df))
fig, ax = plt.subplots(figsize=(7.4, 4.8))
ax.errorbar(
    seed_df["mean_rmse"],
    y_pos,
    xerr=seed_df["std_rmse"],
    fmt="o",
    color="#1F2933",
    ecolor="#7F8C8D",
    elinewidth=1.4,
    capsize=3,
)
ax.set_yticks(y_pos, seed_df["label"])
ax.set_xlabel("Test RMSE")
fig.tight_layout()
seed_path = save_and_display(fig, "results_seed_stability.png", width=820)


Across repeated random seeds, the learned-model ranking is close: fixed transfer has the lowest mean test RMSE (`2.128`), followed by the site-hard selector (`2.135`) and the soft selector (`2.141`). This shows that the single-split advantage of site-hard selection is real in the held-out test split but small relative to seed variability, so it should be presented conservatively.


In [ ]:
apply_thesis_style()
rolling_summary = read_csv(methodology_table_path("rolling_temporal_summary.csv")).copy()
focus_models = [
    "Site-Hard Expert Selector",
    "Context-Aware Selective Learning",
    "Deep Context LSTM (Always TL)",
    "Deep Context LSTM (No TL)",
    "Random Forest",
]
label_map = {
    "Site-Hard Expert Selector": "Site-Hard Selective",
    "Context-Aware Selective Learning": "Soft Selective",
    "Deep Context LSTM (Always TL)": "Always TL",
    "Deep Context LSTM (No TL)": "No TL",
    "Random Forest": "Random Forest",
}
rolling_df = rolling_summary.query("split == 'test' and model_family in @focus_models").copy()
rolling_df["label"] = rolling_df["model_family"].map(label_map)
rolling_df["label"] = pd.Categorical(rolling_df["label"], categories=["Site-Hard Selective", "Soft Selective", "Always TL", "No TL", "Random Forest"], ordered=True)
rolling_df = rolling_df.sort_values("label")

y_pos = np.arange(len(rolling_df))
fig, ax = plt.subplots(figsize=(7.4, 4.8))
ax.errorbar(
    rolling_df["mean_rmse"],
    y_pos,
    xerr=rolling_df["std_rmse"],
    fmt="o",
    color="#1F2933",
    ecolor="#7F8C8D",
    elinewidth=1.4,
    capsize=3,
)
ax.set_yticks(y_pos, rolling_df["label"])
ax.set_xlabel("Test RMSE")
fig.tight_layout()
rolling_path = save_and_display(fig, "results_rolling_temporal_stability.png", width=820)


Across rolling temporal folds, the two selective variants are essentially tied and slightly ahead of fixed transfer: soft selective learning has mean RMSE `1.908`, site-hard selection `1.911`, and fixed transfer `1.915`. The spread across folds is noticeable, so I treat the rolling analysis as support for robustness rather than as proof of a large performance gap.


In [ ]:
apply_thesis_style()
ablation = read_csv(methodology_table_path("selector_ablation_summary.csv")).copy()
ablation = ablation.sort_values("rmse_test", ascending=False)

fig, ax = plt.subplots(figsize=(8.8, 4.9))
ax.barh(ablation["ablation_label"], ablation["rmse_test"], color="#2C7FB8", edgecolor="white", linewidth=1.0)
ax.set_xlabel("Test RMSE")
ax.set_ylabel("")
fig.tight_layout()
ablation_path = save_and_display(fig, "results_selector_ablation.png", width=980)


The ablation results are also more conservative after the leakage fix. The site-hard selector is the best selector variant in this run, while the main soft gate is not the lowest-RMSE selector. I keep the soft gate as the thesis model because it directly represents a continuous no-transfer versus transfer decision, but the ablation table should be read as evidence that selector design remains a material source of uncertainty.


## 4. Source Filtering

This section evaluates the sensitivity of the source-filtering rule used for transfer selection.


In [ ]:
apply_thesis_style()
best_matches = read_csv(eda_table_path("source_target_best_matches_train.csv")).copy()
full_similarity = read_csv(eda_table_path("source_target_similarity_train.csv")).copy()

thresholds = np.round(np.arange(0.20, 0.71, 0.05), 2)
n_wells = int(best_matches["site_id"].nunique())

records = []
for tau in thresholds:
    matched_wells = int((best_matches["cosine_similarity"] >= tau).sum())
    retained_cells = int(full_similarity.loc[full_similarity["cosine_similarity"] >= tau, "cell_id"].nunique())
    records.append(
        {
            "threshold": tau,
            "matched_wells": matched_wells,
            "matched_well_share": matched_wells / n_wells,
            "retained_source_cells": retained_cells,
        }
    )

grid = pd.DataFrame(records)

grid_display = grid.loc[grid['threshold'].isin([0.50, 0.55, 0.60, 0.65]), [
    'threshold', 'retained_source_cells', 'matched_wells', 'matched_well_share'
]].copy()
grid_display['matched_well_share'] = (grid_display['matched_well_share'] * 100).round(1)
grid_display.columns = ['threshold', 'retained_source_cells', 'matched_wells', 'matched_well_share_pct']
display(grid_display)

fig, ax1 = plt.subplots(figsize=(9.0, 4.8))
ax2 = ax1.twinx()
ax1.plot(grid["threshold"], grid["retained_source_cells"], color="#2C7FB8", marker="o", linewidth=2.2, label="Retained source cells")
ax2.plot(grid["threshold"], grid["matched_well_share"] * 100, color="#C97C00", marker="s", linewidth=2.0, label="Target wells with match (%)")
ax1.axvline(0.60, linestyle="--", linewidth=1.0, color="#1F2933", alpha=0.75)
ax1.set_xlabel("Similarity threshold for source filtering")
ax1.set_ylabel("Retained source cells", color="#2C7FB8")
ax2.set_ylabel("Target wells with qualifying match (%)", color="#C97C00")
lines = [
    Line2D([0], [0], color="#2C7FB8", marker="o", linewidth=2.2, label="Retained source cells"),
    Line2D([0], [0], color="#C97C00", marker="s", linewidth=2.0, label="Target wells with match (%)"),
]
ax1.legend(handles=lines, frameon=False, fontsize=8.5, loc="upper right")
fig.tight_layout()
source_filter_path = save_and_display(fig, "results_source_threshold_sensitivity.png", width=960)


The threshold grid shows a steep trade-off. At `0.50`, the filter retains `273` source cells and `9/99` target wells with a direct qualifying match; at `0.60`, it retains `160` source cells and `7/99` wells; at `0.65`, it falls to `103` source cells and `6/99` wells. I therefore treat `maxsim0.60` as a conservative source-filtering choice rather than a universal rule. It removes many weak source cells, while the selector still makes the final transfer decision at the well level.


## 5. Selector Interpretation

Because the selector is the main methodological contribution, this section examines both which features are most influential and how the transfer decision changes with source-target similarity.


In [ ]:
apply_thesis_style()
selector_importance = read_csv(methodology_table_path("selector_feature_importance.csv")).copy()
top_features = selector_importance.nlargest(12, "importance").copy().sort_values("importance", ascending=True)
top_features["feature_label"] = top_features["feature"].map(_humanize_feature)
top_features["feature_group"] = top_features["feature"].map(_feature_group)
top_features["color"] = top_features["feature_group"].map(FEATURE_GROUP_COLORS)

fig, ax = plt.subplots(figsize=(8.8, 5.3))
ax.barh(
    top_features["feature_label"],
    top_features["importance"],
    color=top_features["color"],
    edgecolor="white",
    linewidth=1.0,
)
ax.set_xlabel("Selector feature importance")
feature_handles = [
    Patch(facecolor=color, edgecolor="none", label=group)
    for group, color in FEATURE_GROUP_COLORS.items()
    if group in set(top_features["feature_group"])
]
ax.legend(handles=feature_handles, frameon=False, fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
fig.subplots_adjust(bottom=0.24)
selector_feature_path = save_and_display(fig, "results_selector_feature_importance.png", width=980)


Feature-importance results show that the selector relies most strongly on recent local groundwater depth, seasonality, site-history summaries, and a small group of shared-source signals. I interpret this as evidence that the gate does not choose transfer from similarity alone. It combines local target-state information with broader source-target context.


In [ ]:
apply_thesis_style()
selector_by_site = read_csv(methodology_table_path("selector_by_site.csv")).copy()
selector_by_site["decision_group"] = selector_by_site["selected_parent"].fillna("Deep Context LSTM (No TL)").map(
    {
        "Deep Context LSTM (Always TL)": "Transfer-heavy decision",
        "Deep Context LSTM (No TL)": "No-transfer decision",
        "Random Forest": "Fallback baseline",
    }
).fillna("Fallback baseline")

decision_colors = {
    "Transfer-heavy decision": "#C97C00",
    "No-transfer decision": "#6B7280",
    "Fallback baseline": "#2C7FB8",
}

fig, ax = plt.subplots(figsize=(8.2, 5.2))
for decision_group, sub in selector_by_site.groupby("decision_group", observed=True):
    ax.scatter(
        sub["cosine_similarity"],
        sub["selector_gamma_mean"],
        s=np.clip(sub["months_used"].fillna(sub["months_used"].median()), 5, 110) * 2.5,
        alpha=0.75,
        color=decision_colors[decision_group],
        edgecolor="white",
        linewidth=0.6,
        label=decision_group,
    )

binned = _binned_mean(selector_by_site, "cosine_similarity", "selector_gamma_mean")
if not binned.empty:
    ax.plot(binned["x_mean"], binned["y_mean"], color="#1F2933", linewidth=2.0, linestyle="--")

ax.set_xlabel("Best source-target cosine similarity")
ax.set_ylabel("Mean selector gamma")
ax.legend(frameon=False, fontsize=7.5, loc="upper left")
fig.tight_layout()
selector_decision_path = save_and_display(fig, "results_selector_decision_pattern.png", width=900)


The decision plot reinforces that interpretation. Higher source-target similarity is compatible with more transfer-heavy behaviour, but the overlap across decision groups is substantial. I therefore do not read the selector as a hidden threshold rule. It uses similarity as one cue, while recent local state and site history still shape the final choice.


## 6. Error Analysis

I add error analysis to identify where the best learned model fails, not only which model has the lowest average RMSE. The diagnostics focus on anomaly magnitude, calendar month, site-level failure cases, and whether the site-hard selector chose the better parent on the test period.

In [ ]:
error_regime = read_csv(methodology_table_path("error_by_anomaly_regime.csv")).copy()
error_month = read_csv(methodology_table_path("error_by_calendar_month.csv")).copy()
site_diag = read_csv(methodology_table_path("site_error_diagnostics.csv")).copy()
selector_failure = read_csv(methodology_table_path("selector_failure_typology.csv")).copy()
driver_corr = read_csv(methodology_table_path("error_driver_correlations.csv")).copy()
worst_sites = read_csv(methodology_table_path("worst_site_error_profile.csv")).copy()

key_error_numbers = pd.DataFrame(
    [
        {
            "diagnostic": "extreme_test_observation_share",
            "value": float(site_diag["extreme_share_abs_gt_1_5"].mul(site_diag["n_test"]).sum() / site_diag["n_test"].sum()),
        },
        {
            "diagnostic": "selector_matches_test_best_parent_share",
            "value": float(site_diag["selection_matches_test_best_parent"].mean()),
        },
        {
            "diagnostic": "median_selector_penalty_vs_test_best_parent",
            "value": float(site_diag["rmse_gap_vs_test_best_parent"].median()),
        },
        {
            "diagnostic": "max_site_hard_abs_error",
            "value": float(site_diag["max_abs_error"].max()),
        },
    ]
)
display(key_error_numbers.round(3))
display(driver_corr.head(6).round(3))
display(worst_sites[["site_id", "n_test", "site_hard_rmse", "test_target_std", "max_abs_target", "max_abs_error", "selected_parent", "test_best_parent", "selector_failure_type"]].round(3))


In [ ]:
apply_thesis_style()
focus_models = [
    "Site-Hard Expert Selector",
    "Context-Aware Selective Learning",
    "Deep Context LSTM (Always TL)",
    "Deep Context LSTM (No TL)",
    "Random Forest",
]
label_map = {
    "Site-Hard Expert Selector": "Site-Hard",
    "Context-Aware Selective Learning": "Soft Selective",
    "Deep Context LSTM (Always TL)": "Always TL",
    "Deep Context LSTM (No TL)": "No TL",
    "Random Forest": "Random Forest",
}
regime_order = ["near-normal |y| <= 0.5", "moderate 0.5 < |y| <= 1.5", "extreme |y| > 1.5"]
plot_df = error_regime.query("model_family in @focus_models").copy()
plot_df["model_label"] = plot_df["model_family"].map(label_map)
plot_df["target_abs_regime"] = pd.Categorical(plot_df["target_abs_regime"], categories=regime_order, ordered=True)
plot_df = plot_df.sort_values(["target_abs_regime", "model_label"])

fig, ax = plt.subplots(figsize=(10.2, 5.2))
x = np.arange(len(regime_order))
width = 0.15
for idx, model in enumerate([label_map[m] for m in focus_models]):
    vals = []
    frame = plot_df[plot_df["model_label"].eq(model)].set_index("target_abs_regime")
    for regime in regime_order:
        vals.append(float(frame.loc[regime, "rmse"]) if regime in frame.index else np.nan)
    offset = (idx - 2) * width
    ax.bar(x + offset, vals, width=width, label=model)
ax.set_xticks(x, ["Near-normal", "Moderate", "Extreme"])
ax.set_ylabel("Test RMSE")
ax.legend(ncol=3, frameon=False, fontsize=8)
fig.tight_layout()
error_regime_path = save_and_display(fig, "results_error_by_anomaly_regime.png", width=920)


In [ ]:
apply_thesis_style()
month_focus = ["Site-Hard Expert Selector", "Deep Context LSTM (Always TL)", "Deep Context LSTM (No TL)", "Random Forest"]
month_df = error_month.query("model_family in @month_focus").copy()
fig, ax = plt.subplots(figsize=(9.4, 4.8))
for model in month_focus:
    frame = month_df[month_df["model_family"].eq(model)].sort_values("target_month_of_year")
    ax.plot(frame["target_month_of_year"], frame["rmse"], marker="o", linewidth=1.8, label=label_map.get(model, model))
ax.set_xticks(range(1, 13))
ax.set_xlabel("Target calendar month")
ax.set_ylabel("Test RMSE")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
monthly_error_path = save_and_display(fig, "results_error_by_calendar_month.png", width=900)


In [ ]:
apply_thesis_style()
pretty_driver = {
    "test_target_std": "Target variability",
    "max_abs_target": "Max |target|",
    "mean_abs_target": "Mean |target|",
    "extreme_share_abs_gt_1_5": "Extreme share",
    "cosine_similarity": "Source similarity",
    "wtda_std": "Train local variability",
    "sim_top3_mean": "Top-3 similarity",
    "max_observations": "Historical observations",
    "selector_gamma_mean": "Mean soft gate",
    "train_rows": "Train rows",
    "months_used": "Matched months",
}
plot_corr = driver_corr.head(8).copy().sort_values("spearman_rho_with_site_hard_rmse")
plot_corr["label"] = plot_corr["driver"].map(pretty_driver).fillna(plot_corr["driver"])
colors = ["#0B6E4F" if value >= 0 else "#C97C00" for value in plot_corr["spearman_rho_with_site_hard_rmse"]]
fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.barh(plot_corr["label"], plot_corr["spearman_rho_with_site_hard_rmse"], color=colors)
ax.axvline(0, color="#1F2933", linewidth=0.9)
ax.set_xlabel("Spearman correlation with site RMSE")
fig.tight_layout()
driver_path = save_and_display(fig, "results_error_driver_correlations.png", width=880)


In [ ]:
apply_thesis_style()
worst_plot = worst_sites.head(10).copy().sort_values("site_hard_rmse")
color_map = {
    "selected_test_best_parent": "#0B6E4F",
    "selected_test_worse_parent": "#C0392B",
    "parent_tie_on_test": "#6B7280",
}
colors = [color_map.get(value, "#6B7280") for value in worst_plot["selector_failure_type"]]
fig, ax = plt.subplots(figsize=(9.2, 5.0))
ax.barh(worst_plot["site_id"].astype(str), worst_plot["site_hard_rmse"], color=colors)
ax.set_xlabel("Site-Hard test RMSE")
ax.set_ylabel("Site ID")
legend_items = [Patch(facecolor=color, label=label.replace("_", " ")) for label, color in color_map.items()]
ax.legend(handles=legend_items, frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
worst_path = save_and_display(fig, "results_worst_site_error_profile.png", width=900)


In [ ]:
apply_thesis_style()
failure_plot = selector_failure.copy()
failure_plot["route"] = (
    failure_plot["selected_parent"].str.replace("Deep Context LSTM ", "", regex=False)
    + " -> "
    + failure_plot["test_best_parent"].str.replace("Deep Context LSTM ", "", regex=False)
)
failure_plot = failure_plot.sort_values("sites", ascending=True)
fig, ax = plt.subplots(figsize=(9.2, 4.8))
colors = ["#0B6E4F" if value == "selected_test_best_parent" else "#C0392B" for value in failure_plot["selector_failure_type"]]
ax.barh(failure_plot["route"], failure_plot["sites"], color=colors)
ax.set_xlabel("Number of wells")
ax.set_ylabel("")
fig.tight_layout()
failure_path = save_and_display(fig, "results_selector_failure_typology.png", width=900)


The error analysis shows that the average result is dominated by anomaly magnitude. For the site-hard selector, RMSE is `0.474` for near-normal targets, `0.820` for moderate anomalies, and `3.883` for extreme anomalies. Extreme test observations are only `25.4%` of the test set, but they dominate the pooled squared error.

The strongest site-level error driver is target variability itself: site-level RMSE correlates with test-period target standard deviation at Spearman `rho = 0.935`, while source-target cosine similarity is much weaker (`rho = -0.197`, `p = 0.054`). The worst single site is `31247`, where the maximum target anomaly reaches `28.4` and the maximum absolute prediction error is `24.65`. This is why the thesis should report site-cluster robustness and worst-site diagnostics, not only pooled RMSE.

The site-hard selector chooses the test-best parent for `57/99` wells and the worse parent for `42/99` wells. Among the mismatched wells, the median penalty relative to the test-best parent is about `0.041` site RMSE, which explains why the selector can edge fixed transfer on pooled RMSE while still showing only a small and statistically fragile advantage over fixed transfer.

## 7. Discussion And Limitations

After the leakage fix, the main learned-model result supports a narrow selective-transfer claim. The site-hard selector gives the lowest single-split test RMSE, but the margin over fixed transfer is very small. I therefore avoid an over-broad superiority claim and frame the contribution as controlled transfer selection under heterogeneous well behaviour.

The useful contribution is that transfer is not uniformly helpful across wells. A selector can identify part of that heterogeneity and reduce some negative-transfer damage when fixed transfer is harmful. The negative-transfer analysis remains central because it explains why selective transfer is needed rather than treating transfer learning as automatically beneficial.

I still interpret these findings within three main limits. First, the Amsterdam target set is restricted, so I do not claim automatic generalisation to other Dutch regions or other hydrogeological settings. Second, the source and target domains are only partially aligned, which is exactly why a conservative source filter and a selector are necessary in this study. Third, monthly aggregation smooths short-term dynamics, so short-horizon groundwater memory remains an important challenge for learned models.

The robustness checks reduce the risk that the learned-model ranking is driven by one seed or one evaluation window, but they do not replace external validation on a new region. I would frame the thesis as evidence that selective transfer can diagnose and mitigate negative transfer in this Amsterdam case study.
